# **Welcome to CATNIP**

There are 2 pieces to CATNIP, the python modules and the google sheet. First, make sure your Google Sheet is set up according to the README. This is where important disk information is stored that is read by the python modules. Once your Google Sheet is ready, we can begin.

## Setting up paths

In [ ]:
# Directory where python modules are located
moduledir = 'path to modules'

# Directory where your .fits files (images) are located
datadir = 'path to image data files'

# URL of your spreadsheet interface
wburl = 'url of spreadsheet'

## Importing CATNIP modules

In [ ]:
# Import modules
import sys
sys.path.append(moduledir)
import sheetreader
import DiskAnalysisPipeline
from imp import reload
reload(sheetreader)
reload(DiskAnalysisPipeline)

import os
os.chdir(datadir)

# you may need to pip install gspread oauth2client and imp

## Obtaining credentials to access the Google Sheet

In [ ]:
# Authorizing google credentials to open google sheet

import gspread
from oauth2client.service_account import ServiceAccountCredentials

# Define the scope and authorize
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("YOURFILENAME.json", scope)
gc = gspread.authorize(creds)


## Reading the Google Sheet

In [ ]:
# Read sheet and save the data and display dicts
wb = gc.open_by_url(wburl)
master_dict = sheetreader.wb_to_dict(wb, 'Image Data', add_paths=False, settings_sheet='Image Settings', namekey='Tracer', splitkey='Object')
master_display_dict = sheetreader.wb_to_dict(wb, 'Disk Information', settings_sheet='Disk Settings')

## Creating a dictionary for your disks

In [ ]:
astro_objects = {}
disk_lst = [disk for disk in master_display_dict if master_display_dict[disk]['Use?']]
for disk in disk_lst:
    astro_objects[disk] = catnip.AstroObject(disk, master_dict[disk], master_display_dict[disk])

In [ ]:
# Use this cell to update display dict info without reprocessing so 
# you don't have to run the whole notebook over again
wb = gc.open_by_url(wburl)
master_display_dict = sheetreader.wb_to_dict(wb, 'Disk Information', settings_sheet='Disk Settings')
disk_lst = [disk for disk in master_display_dict if master_display_dict[disk]['Use?']]
for disk in disk_lst: astro_objects[disk].update(master_display_dict[disk])

In [ ]:
# Plot profiles and images for your desired disks
for disk in disk_lst: astro_objects[disk].make_plot_multiline()